In [ ]:
#------------------------------------------------------------
# Connecting to an openLCA database:
#------------------------------------------------------------
import olca_ipc as ipc
import olca_schema as o   # Entity data structures are handled by the olca_schema package (a sub-package of olca_ipc)
client = ipc.Client(8080)
print("Connected:", client)

Connected: <olca_ipc.ipc.Client object at 0x0000024726E6EF50>
Python Type: <class 'olca_schema.schema.Process'>
ID: 9f5e9157-4560-a9da-d51b-9f8c7ff3670b
Python Type: <class 'olca_schema.schema.Process'>
ID: 0bd3aec8-286b-466f-868c-87917428450c
Name: Electricity grid mix 1kV-60kV, consumption mix, at consumer, AC, 1kV - 60kV

Attributes of an o.Process Instance (Contains all details about the process:):
['id', 'allocation_factors', 'category', 'default_allocation_method', 'description', 'dq_entry', 'dq_system', 'exchange_dq_system', 'exchanges', 'is_infrastructure_process', 'last_change', 'last_internal_id', 'library', 'location', 'name', 'parameters', 'process_documentation', 'process_type', 'social_aspects', 'social_dq_system', 'tags', 'version']

Attributes of an o.Ref Instance (Contains only a subset of information:):
['id', 'category', 'description', 'flow_type', 'library', 'location', 'name', 'process_type', 'ref_unit', 'ref_type']
1: CUTOFF gypsum; at point-of-sale
2: CUTOFF pest

Accesing database elements

In [ ]:
# Usando client.get() con ID
process_via_id = client.get(
    o.Process, 
    "9f5e9157-4560-a9da-d51b-9f8c7ff3670b"
)

print("Using get() with ID:")
print("Python Type:", type(process_via_id))
print("ID:", process_via_id.id)
print("Name:", process_via_id.name)

# Usando client.find() para obtener referencia y luego get() para la entidad completa
process_via_find = client.find(
    o.Process, 
    name="Electricity grid mix 1kV-60kV, consumption mix, at consumer, AC, 1kV - 60kV"
)

process_via_get = client.get(o.Process, process_via_find.id)

print("\nUsing find() + get():")
print("Python Type:", type(process_via_get))
print("ID:", process_via_get.id)
print("Name:", process_via_get.name)

# Diferencia entre Entity y Reference
print("\nAttributes of an o.Process Instance (full entity):")
print(list(process_via_id.__dict__.keys()))

print("\nAttributes of an o.Ref Instance (reference):")
print(list(process_via_find.__dict__.keys()))

# Mostrar los primeros 5 proveedores
print("\nFirst 5 providers:")
for i, p in enumerate(client.get_providers()[:5]):
    print(f"{i+1}: {p.flow.name}")
print("...")

Creating database elements

In [ ]:
#New Flow Property
flow_property_name = "Mass (Demo)"
unit_symbol = "kg"
unit_group = o.new_unit_group(f"{flow_property_name} units", unit_symbol)
#Creation of the flow property:
prop = o.new_flow_property(flow_property_name, unit_group)

# client.put_all() can add multiple entities to the database:
client.put_all(
    unit_group,
    prop,
)
#---------------------------------------
#New Flow 
is_elementary = True #True is elementary flow, False is product flow
flow_name = "Biomass (Demo)"

# To illustrate the two different types of flows:
if is_elementary:#if is elementary, create an elementary flow
    flow = o.new_elementary_flow(flow_name, prop)
    flow.category = "Demonstration/Elementary"
else:#if is not elementary, create a product flow, which can have providers 
    flow = o.new_product(flow_name, prop) # Can have providers (technosphere flows)
    flow.category = "Demonstration/Products"

# client.put() adds a single entity to the database:
client.put(flow)

#---------------------------------------
#New process
process_name = "Biomass to Energy (Demo)"
process = o.new_process(process_name)
## Inputs:
# As input we can use the Biomass flow we created before:
o.new_input(process, flow, 1000) 
# We can also use an existing flow from our database,
# for example the transportion of biomass (CUTOFF bulk cargo rail transport; diesel):
o.new_input(process, client.get(o.Flow, name='CUTOFF bulk cargo rail transport; diesel'), 1000)

## Outputs:
# First, let's define the main output of the process:
# New Unit Property:
name = "Energy (Demo)"
unit_symbol = "MJ"
unit_group = o.new_unit_group(f"{name} units", unit_symbol)
prop = o.new_flow_property(name, unit_group)
# New Flow:
main_output_flow = o.new_product('Energy (from Biomass, Demo)', prop)
main_output_flow.category = "Demonstration/Products"
# New Output:
main_output = o.new_output(process, main_output_flow, 15000)
main_output.is_quantitative_reference = True # To mark a flow as the main output, we need to set is_quantitative_reference as True:

# Then we can use some additional existing outputs from the database,
o.new_output(process, client.get(o.Flow, name='Water in slurry'), 1800)
o.new_output(process, client.get(o.Flow, name='polyamide 6.6 fibres (PA 6.6)'), 10)


process.category = "Demonstration" 
client.put_all(
    unit_group,
    prop,
    main_output_flow,
    process,
)

#---------------------------------------
#New Product system
#Automaticamente crea el product system
# Configuration for auto-completion of product systems:
config = o.LinkingConfig(
    prefer_unit_processes=True,
    provider_linking=o.ProviderLinking.PREFER_DEFAULTS,
)
# A product system is build from a process:
process_ref = client.find(o.Process, "Biomass to Energy (Demo)")
system_ref = client.create_product_system(process_ref, config)
print(f"Created product system {system_ref.name}, id={system_ref.id}")

AttributeError: 'NoneType' object has no attribute 'to_ref'

Providers and parameters

In [ ]:
flow_ref = o.Ref(id="de60e9f5-87a0-40cb-ae54-11ec7eac8909") # ID of the CUTTOFF bulk cargo rail transport; diesel flow
providers = client.get_providers(
    flow=flow_ref
)

print('Flow:', client.get(o.Flow, flow_ref.id).name)
print('\nProviders:')
tech_flows = {}
for i, p in enumerate(providers):
    print(f"{i+1}: {p.provider.name}")
    # Save the flows for later:
    tech_flows[p.provider.name] = p

#They do much more things (check the guidelances for more details)

Flow: ammonia, liquid, at regional storehouse

Providers:


Impact methods

In [36]:
# Get all impact method descriptors
methods = client.get_descriptors(o.ImpactMethod)

# Loop through all methods with numbering
for i, method in enumerate(methods, start=1):
    print(f"{i}: {method.name}  {method.id}")
# Get all impact method descriptors
methods = client.get_descriptors(o.ImpactMethod)

1: USEtox Recommended + Interim  0c15a246-fc4e-3d6f-b6df-81eaabb315af
2: USEtox Recommended  6f23da2f-a6d0-353d-b2e1-18cdd91cac4c
3: USEtox 2 (recommended only)  af11ef29-8b3a-37ba-acc5-a7d01a871386
4: USEtox 2 (recommended + interim)  5eb764ea-3f85-33f2-b99c-fc8c5b832f99
5: USEtox (sensitivity)  5c1d2182-af2a-3482-84e2-6cb75b99f019
6: USEtox (recommended + interim)  07c93022-3762-3caa-a357-d95fc26c7bd5
7: USEtox (default)  5ca5cc21-755b-34b4-a03d-4c0b1cd1904f
8: USEtox (consensus only)  4e4669e7-98fd-386a-a890-4ec17dc773bf
9: TRACI 2.1  d2c781ce-21b4-3218-8fca-78133f2c8d4d
10: TRACI 2  3bdedfd5-6825-36ff-8c1f-494ce0d99ad1
11: TRACI  a36e90a3-5419-3b08-8244-d18d8b029226
12: Selected LCI results, additional  09e9337a-b9ee-3e88-aaaf-bbff90744119
13: Selected LCI results  3a748812-7a13-3d2a-b663-6b62f7878bf9
14: ReCiPe Midpoint (I)  a816b9c4-aef5-3aba-86c1-f74f87ee5f8f
15: ReCiPe Midpoint (H)  4389133f-3cf9-32bd-ba12-3d2801bcfcb4
16: ReCiPe Midpoint (E)  4144d9ce-a427-33ab-8a52-1a81090ac0

Calculate Impact Example

In [ ]:
#Calculate a product system with a specific method:
system_ref = client.find(o.ProductSystem, '50/50 polyester-cotton terry towel production and use')
method_ref = client.find(o.ImpactMethod, 'ReCiPe 2016 Endpoint (H)')
setup = o.CalculationSetup(
    target=system_ref,
    impact_method=method_ref,
)

# Start Calculation:
result = client.calculate(setup)

# Once the result is ready we can access it:
result.wait_until_ready()
impacts = result.get_total_impacts()
for i in impacts:
    assert i.impact_category
    print(f"{i.impact_category.name} {i.amount} {i.impact_category.ref_unit}")

# It is very important to call result.dispose()! (to free up memory)
result.dispose()

#---------------------------------------------
#Calculate Impacts for Different Parameters(i do not have that paramenters defined but it is possible):
"""""""""
import pandas as pd
param_names = ['c0', 'c_1', 'c_2', 'c_3', 'c_4']
all_model_params = client.get_parameters(o.ProductSystem, system_ref.id)
params_dict = {p.name: p for p in all_model_params if p.name in param_names}
method_ref = client.find(o.ImpactMethod, 'IPCC 2013 GWP 100a (incl. CO2 uptake)')

results_dict = {}
for cur_p_name in param_names:
    
    parameters = []
    for p_n, p in params_dict.items():
        # Assign 100% to the current option:
        if p_n == cur_p_name:
            parameters.append(o.ParameterRedef(name=p.name, value=1.0, context=p.context))
        # Other options get 0%:
        else:
            parameters.append(o.ParameterRedef(name=p.name, value=0.0, context=p.context))
    
    setup = o.CalculationSetup(
        target=system_ref,
        impact_method=method_ref,
        parameters=parameters
    )
    result = client.calculate(setup)
    assert result
    result.wait_until_ready()
    impacts = result.get_total_impacts()

    results_dict[cur_p_name] = {}
    for i in impacts:
        results_dict[cur_p_name][f"{i.impact_category.name} ({i.impact_category.ref_unit})"] = i.amount 
    result.dispose()

pd.DataFrame(results_dict)
"""


Water consumption, Aquatic ecosystems - ReCiPe 2016 Endpoint (H) 8.727571066004007e-15 species.yr
Ozone formation, Human health - ReCiPe 2016 Endpoint (H) 7.185360362814096e-09 DALY
Terrestrial ecotoxicity - ReCiPe 2016 Endpoint (H) 1.6319879510852354e-11 species.yr
Water consumption, Human health - ReCiPe 2016 Endpoint (H) 3.2078158553855794e-08 DALY
Global warming, Freshwater ecosystems - ReCiPe 2016 Endpoint (H) 2.19428871359345e-13 species.yr
Terrestrial acidification - ReCiPe 2016 Endpoint (H) 2.981507062441397e-09 species.yr
Global warming, Terrestrial ecosystems - ReCiPe 2016 Endpoint (H) 8.031617612555774e-09 species.yr
Stratospheric ozone depletion - ReCiPe 2016 Endpoint (H) 2.7362903523428666e-10 DALY
Ozone formation, Terrestrial ecosystems - ReCiPe 2016 Endpoint (H) 1.02379866536068e-09 species.yr
Ionizing radiation - ReCiPe 2016 Endpoint (H) 1.718221333878337e-09 DALY
Fossil resource scarcity - ReCiPe 2016 Endpoint (H) 0.0 USD2013
Global warming, Human health - ReCiPe 2016 

Inventory calculation  

In [48]:

import pandas as pd

setup = o.CalculationSetup(
    target=system_ref,
    #unit=count.unit_group.ref_unit,  # "Item(s)"
    #amount=10,
)
result = client.calculate(setup)
result.wait_until_ready()

inventory = result.get_total_flows()
inventory_df = (
    pd.DataFrame(
        data=[
            (
                i.envi_flow.flow.name,
                i.envi_flow.is_input,
                i.amount,
                i.envi_flow.flow.ref_unit,
            )
            for i in inventory
        ],
        columns=["Flow", "Is input?", "Amount", "Unit"],
    )
)

result.dispose() # important!

inventory_df[inventory_df['Amount'] != 0.0]

,Flow,Is input?,Amount,Unit
0,"ammonium nitrate, as N, at regional storehouse",True,9.501030e-03,kg
1,"acetamide-anillide-compounds, at regional stor...",True,9.504000e-04,kg
2,Uranium,True,1.184244e+01,MJ
3,"AOX, Adsorbable Organic Halogen as Cl",False,1.197273e-06,kg
4,Cadmium,False,2.028066e-07,kg
...,...,...,...,...
540,cottonseed; at harvest in 2007; at farm; 90%-9...,True,1.188000e-03,kg
541,"CUTOFF irrigate; gravity, groundwater source, ...",True,1.318680e+00,m3
542,"Carbon dioxide, fossil",False,1.372140e-01,kg
543,"Methane, fossil",False,5.940000e-06,kg


In [44]:
import sys
!{sys.executable} -m pip install pandas

  Using cached pandas-3.0.1-cp314-cp314-win_amd64.whl.metadata (19 kB)
  Using cached numpy-2.4.2-cp314-cp314-win_amd64.whl.metadata (6.6 kB)
Using cached pandas-3.0.1-cp314-cp314-win_amd64.whl (9.9 MB)
Using cached numpy-2.4.2-cp314-cp314-win_amd64.whl (12.4 MB)

   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ---------------------------------------- 0/2 [numpy]
   ----

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
